# Extracción de datos TikTok - Lululemon por Hashtags

Se buscan vídeos de TikTok relacionados con Lululemon a partir de hashtags.
Se extraen los vídeos con más engagement y luego se descargan los comentarios de los top vídeos.

**Dos zonas geográficas** para análisis comparativo:
- Estados Unidos (US) — comentarios en inglés
- Europa (ES, FR, GB, DE) — comentarios filtrados por idioma del país

**Filtrado por idioma:**
- US (proxy US) → solo inglés
- España (proxy ES) → solo español
- Francia (proxy FR) → solo francés
- Reino Unido (proxy GB) → solo inglés
- Alemania (proxy DE) → solo alemán

**Actores de Apify:**
- `apidojo/tiktok-scraper` — búsqueda por hashtags con filtro de `location`
- `clockworks/tiktok-comments-scraper` — extracción de comentarios

In [1]:
pip install apify-client pandas langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993332 sha256=fbb9dbfb1852be86500b95ecf2f86a370a30c52374b6c64803716e2751cd24d6
  Stored in directory: /Users/gonzaloalonsolidon/Library/Caches/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from apify_client import ApifyClient
import pandas as pd
import json
from langdetect import detect, LangDetectException

from dotenv import load_dotenv
import os

load_dotenv()
APIFY_TOKEN = os.getenv("APIFY_TOKEN")
client = ApifyClient(APIFY_TOKEN)

In [ ]:
# Hashtags relacionados con Lululemon
HASHTAGS = [
    "lululemon",
    "lululemonhaul",
    "lululemonreview",
    "lululemonfavorites",
    "lululemonoutfit",
    "lululemonleggings",
    "lululemontryhaul",
    "lululemonfinds",
]

HASHTAG_URLS = [{"url": f"https://www.tiktok.com/tag/{h}"} for h in HASHTAGS]
print(f"Total hashtags: {len(HASHTAG_URLS)}")
for u in HASHTAG_URLS:
    print(f"  {u['url']}")

Total hashtags: 8
  https://www.tiktok.com/tag/lululemon
  https://www.tiktok.com/tag/lululemonhaul
  https://www.tiktok.com/tag/lululemonreview
  https://www.tiktok.com/tag/lululemonfavorites
  https://www.tiktok.com/tag/lululemonoutfit
  https://www.tiktok.com/tag/lululemonleggings
  https://www.tiktok.com/tag/lululemontryhaul
  https://www.tiktok.com/tag/lululemonfinds


---
## 1. Funciones auxiliares

In [4]:
def build_post_url(row):
    """Construye la URL del post a partir de channel.username e id."""
    channel = row.get("channel")
    video_id = row.get("id")
    if isinstance(channel, dict) and channel.get("username") and video_id:
        return f"https://www.tiktok.com/@{channel['username']}/video/{video_id}"
    return None

def procesar_videos(items):
    """Convierte los items del actor a DataFrame con score y ranking."""
    df = pd.DataFrame(items)
    if len(df) == 0:
        return df
    for col in ["likes", "comments", "shares", "views", "bookmarks"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    df["post_url"] = df.apply(build_post_url, axis=1)
    df["author"] = df["channel"].apply(
        lambda c: c.get("username", "") if isinstance(c, dict) else ""
    )
    df["score"] = df["comments"] * 3 + df["likes"] * 1 + df["shares"] * 2
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    df["rank"] = range(1, len(df) + 1)
    return df

def detectar_idioma(texto):
    """Detecta el idioma de un texto. Devuelve código ISO o 'unknown'."""
    try:
        if not texto or not isinstance(texto, str) or len(texto.strip()) < 3:
            return "unknown"
        return detect(texto)
    except LangDetectException:
        return "unknown"

def filtrar_comentarios_por_idioma(df_comments, idiomas_permitidos):
    """Filtra comentarios por idioma detectado."""
    # Buscar columna de texto del comentario
    col_texto = None
    for c in ["text", "comment", "commentText", "body"]:
        if c in df_comments.columns:
            col_texto = c
            break
    if col_texto is None:
        print("AVISO: No se encontró columna de texto. Columnas:", df_comments.columns.tolist())
        return df_comments
    
    df_comments["idioma_detectado"] = df_comments[col_texto].apply(detectar_idioma)
    
    print(f"Distribución de idiomas detectados:")
    print(df_comments["idioma_detectado"].value_counts().head(10))
    
    df_filtrado = df_comments[df_comments["idioma_detectado"].isin(idiomas_permitidos)].copy()
    print(f"\nComentarios antes del filtro: {len(df_comments)}")
    print(f"Comentarios después del filtro ({', '.join(idiomas_permitidos)}): {len(df_filtrado)}")
    return df_filtrado

print("Funciones cargadas.")

Funciones cargadas.


---
## 2. Extracción de vídeos - Estados Unidos

`location: "US"` — luego los comentarios se filtrarán a solo inglés.

In [5]:
run_us = client.actor("apidojo/tiktok-scraper").call(
    run_input={
        "startUrls": HASHTAG_URLS,
        "maxItems": 200,
        "location": "US",
    }
)

print("Run ID (US):", run_us["id"])
print("Dataset ID (US):", run_us["defaultDatasetId"])

[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:34.469Z ACTOR: Pulling container image of build 1wgxHJCb5SrQmUuTk from registry.
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:34.472Z ACTOR: Creating container.
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:34.525Z ACTOR: Starting container.
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:35.918Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:35.956Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:36.263Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tiktok-scraper runId:TjOOACa1Qp9kGXXbu] -> 2026-03-26T12:42:36.264Z INFO  [Status message]: Your run has started.
[apify.tiktok-scra

Run ID (US): TjOOACa1Qp9kGXXbu
Dataset ID (US): sipR2nq5Gi8QAXZPq


In [6]:
items_us = list(client.dataset(run_us["defaultDatasetId"]).iterate_items())
print(f"Total vídeos US: {len(items_us)}")

df_us = procesar_videos(items_us)

print(f"\nTop 10 US por engagement:")
df_us[["rank", "author", "likes", "comments", "shares", "views", "score", "post_url"]].head(10)

Total vídeos US: 200

Top 10 US por engagement:


,rank,author,likes,comments,shares,views,score,post_url
0,1,huntercamp33,608149,482,1850,5371145,613295,https://www.tiktok.com/@huntercamp33/video/720...
1,2,lisi.shops,579366,2092,7022,5543233,599686,https://www.tiktok.com/@lisi.shops/video/72669...
2,3,tiktokwithtay,404703,236,207,7691473,405825,https://www.tiktok.com/@tiktokwithtay/video/69...
3,4,8kzayy,257312,2692,49366,1612464,364120,https://www.tiktok.com/@8kzayy/video/761988239...
4,5,bellagiannop,331023,1589,934,2448898,337658,https://www.tiktok.com/@bellagiannop/video/711...
5,6,rose32xz,288127,856,1306,4841820,293307,https://www.tiktok.com/@rose32xz/video/7552319...
6,7,courtneyycahoon,282257,655,2631,1533658,289484,https://www.tiktok.com/@courtneyycahoon/video/...
7,8,sophiasouzas,237276,430,3301,2301334,245168,https://www.tiktok.com/@sophiasouzas/video/734...
8,9,tawfiktwins,183022,464,27234,1616935,238882,https://www.tiktok.com/@tawfiktwins/video/7438...
9,10,lojanoska,222030,1084,2433,1110202,230148,https://www.tiktok.com/@lojanoska/video/729884...


---
## 3. Extracción de vídeos - Europa

Búsquedas separadas por país: GB, ES, FR, DE. Se marca cada vídeo con su país de origen.

In [7]:
EU_COUNTRIES = [
    ("GB", "Reino Unido"),
    ("ES", "España"),
    ("FR", "Francia"),
    ("DE", "Alemania"),
]

items_europe = []

for code, nombre in EU_COUNTRIES:
    print(f"\nBuscando en {nombre} ({code})...")
    run_eu = client.actor("apidojo/tiktok-scraper").call(
        run_input={
            "startUrls": HASHTAG_URLS,
            "maxItems": 50,
            "location": code,
        }
    )
    items_country = list(client.dataset(run_eu["defaultDatasetId"]).iterate_items())
    for item in items_country:
        item["location_search"] = code
    items_europe.extend(items_country)
    print(f"  {nombre}: {len(items_country)} vídeos")

print(f"\nTotal vídeos Europa: {len(items_europe)}")


Buscando en Reino Unido (GB)...


[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:51.765Z ACTOR: Pulling container image of build 1wgxHJCb5SrQmUuTk from registry.
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:51.767Z ACTOR: Creating container.
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:51.807Z ACTOR: Starting container.
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:53.381Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:53.441Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:53.798Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tiktok-scraper runId:xcCQsR9uyj1ngak1w] -> 2026-03-26T12:42:53.798Z INFO  [Status message]: Your run has started.
[apify.tiktok-scra

  Reino Unido: 50 vídeos

Buscando en España (ES)...


[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:08.843Z ACTOR: Pulling container image of build 1wgxHJCb5SrQmUuTk from registry.
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:08.846Z ACTOR: Creating container.
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:08.945Z ACTOR: Starting container.
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:11.100Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:11.150Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:11.457Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tiktok-scraper runId:FIzY2pOAvQ3XHqj9C] -> 2026-03-26T12:43:11.458Z INFO  [Status message]: Your run has started.
[apify.tiktok-scra

  España: 50 vídeos

Buscando en Francia (FR)...


[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:25.833Z ACTOR: Pulling container image of build 1wgxHJCb5SrQmUuTk from registry.
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:25.836Z ACTOR: Creating container.
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:25.885Z ACTOR: Starting container.
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:27.311Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:27.373Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:27.690Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tiktok-scraper runId:a4EIT8iCfWd4ooWeD] -> 2026-03-26T12:43:27.691Z INFO  [Status message]: Your run has started.
[apify.tiktok-scra

  Francia: 50 vídeos

Buscando en Alemania (DE)...


[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:47.501Z ACTOR: Pulling container image of build 1wgxHJCb5SrQmUuTk from registry.
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:47.503Z ACTOR: Creating container.
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:47.593Z ACTOR: Starting container.
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:49.311Z INFO  System info {"apifyVersion":"3.4.2","apifyClientVersion":"2.12.4","crawleeVersion":"3.13.5","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:49.364Z INFO  PHASE -- STARTING ACTOR.
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:49.819Z INFO  PHASE -- SETTING UP CRAWLER.
[apify.tiktok-scraper runId:QS27oOmPAQQxYyx86] -> 2026-03-26T12:43:49.820Z INFO  [Status message]: Your run has started.
[apify.tiktok-scra

  Alemania: 50 vídeos

Total vídeos Europa: 200


In [8]:
df_europe = procesar_videos(items_europe)

print(f"Total vídeos Europa: {len(df_europe)}")
if "location_search" in df_europe.columns:
    print(f"\nPor país de búsqueda:")
    print(df_europe["location_search"].value_counts())

print(f"\nTop 10 Europa por engagement:")
cols = ["rank", "author", "likes", "comments", "shares", "views", "score", "post_url"]
cols = [c for c in cols if c in df_europe.columns]
df_europe[cols].head(10)

Total vídeos Europa: 200

Por país de búsqueda:
location_search
FR    50
DE    50
GB    50
ES    50
Name: count, dtype: int64

Top 10 Europa por engagement:


,rank,author,likes,comments,shares,views,score,post_url
0,1,gabriella.a0,1242747,7130,31181,15533856,1326499,https://www.tiktok.com/@gabriella.a0/video/720...
1,2,gabriella.a0,1242394,7057,31183,15526016,1325931,https://www.tiktok.com/@gabriella.a0/video/720...
2,3,gabriella.a0,1242393,7057,31183,15526014,1325930,https://www.tiktok.com/@gabriella.a0/video/720...
3,4,alissa09082,881054,2852,20272,6236833,930154,https://www.tiktok.com/@alissa09082/video/7615...
4,5,alissa09082,881048,2852,20272,6236681,930148,https://www.tiktok.com/@alissa09082/video/7615...
5,6,itstookate,457639,5643,30343,3711479,535254,https://www.tiktok.com/@itstookate/video/76156...
6,7,itstookate,457639,5643,30343,3711495,535254,https://www.tiktok.com/@itstookate/video/76156...
7,8,athletikatyootd,451555,1001,1676,3515634,457910,https://www.tiktok.com/@athletikatyootd/video/...
8,9,bellagiannop,331023,1589,934,2448898,337658,https://www.tiktok.com/@bellagiannop/video/711...
9,10,bellagiannop,331023,1589,934,2448898,337658,https://www.tiktok.com/@bellagiannop/video/711...


---
## 4. Guardar vídeos y seleccionar top para extraer comentarios

In [9]:
df_us.to_csv("../datos/processed/tiktok/tiktok_lululemon_videos_US.csv", index=False, encoding="utf-8-sig")
df_europe.to_csv("../datos/processed/tiktok/tiktok_lululemon_videos_EUROPE.csv", index=False, encoding="utf-8-sig")

print("Guardados:")
print("  ../datos/processed/tiktok/tiktok_lululemon_videos_US.csv")
print("  ../datos/processed/tiktok/tiktok_lululemon_videos_EUROPE.csv")

Guardados:
  ../datos/processed/tiktok/tiktok_lululemon_videos_US.csv
  ../datos/processed/tiktok/tiktok_lululemon_videos_EUROPE.csv


In [10]:
N_TOP = 10

urls_us = df_us.head(N_TOP)["post_url"].dropna().tolist()
urls_europe = df_europe.head(N_TOP)["post_url"].dropna().tolist()

print(f"URLs seleccionadas US ({len(urls_us)}):")
for i, u in enumerate(urls_us, 1):
    print(f"  {i}. {u}")

print(f"\nURLs seleccionadas Europa ({len(urls_europe)}):")
for i, u in enumerate(urls_europe, 1):
    print(f"  {i}. {u}")

URLs seleccionadas US (10):
  1. https://www.tiktok.com/@huntercamp33/video/7200918254892649770
  2. https://www.tiktok.com/@lisi.shops/video/7266988919261285678
  3. https://www.tiktok.com/@tiktokwithtay/video/6968899199773576453
  4. https://www.tiktok.com/@8kzayy/video/7619882392978803981
  5. https://www.tiktok.com/@bellagiannop/video/7119551158422719787
  6. https://www.tiktok.com/@rose32xz/video/7552319724474912031
  7. https://www.tiktok.com/@courtneyycahoon/video/7173826000139144494
  8. https://www.tiktok.com/@sophiasouzas/video/7343234587000114475
  9. https://www.tiktok.com/@tawfiktwins/video/7438318483160501550
  10. https://www.tiktok.com/@lojanoska/video/7298843917066194218

URLs seleccionadas Europa (10):
  1. https://www.tiktok.com/@gabriella.a0/video/7209474725054090539
  2. https://www.tiktok.com/@gabriella.a0/video/7209474725054090539
  3. https://www.tiktok.com/@gabriella.a0/video/7209474725054090539
  4. https://www.tiktok.com/@alissa09082/video/7615453695496441102

---
## 5. Extracción de comentarios - Estados Unidos

Se extraen comentarios y luego se filtran a **solo inglés**.

In [11]:
MAX_COMMENTS = 200
MAX_REPLIES = 10

print(f"Extrayendo comentarios de {len(urls_us)} vídeos US...")

run_comments_us = client.actor("clockworks/tiktok-comments-scraper").call(
    run_input={
        "postURLs": urls_us,
        "maxComments": MAX_COMMENTS,
        "maxRepliesPerComment": MAX_REPLIES,
    }
)

print("Run ID:", run_comments_us["id"])
print("Dataset ID:", run_comments_us["defaultDatasetId"])

Extrayendo comentarios de 10 vídeos US...


[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> Status: RUNNING, Message: 
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.305Z ACTOR: Pulling container image of build ESKgmPC1krhzsvwG3 from registry.
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.307Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.371Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.373Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.554Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:FZ51cpF3fJqWOzO3H] -> 2026-03-26T12:44:03.556Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:

Run ID: FZ51cpF3fJqWOzO3H
Dataset ID: OOGNuLDsJhkPPOaHg


In [12]:
comments_us_raw = list(client.dataset(run_comments_us["defaultDatasetId"]).iterate_items())
print(f"Total comentarios US (sin filtrar): {len(comments_us_raw)}")

df_comments_us_raw = pd.DataFrame(comments_us_raw)

# Filtrar: solo inglés para US
print("\n--- Filtro de idioma US: solo inglés ---")
df_comments_us = filtrar_comentarios_por_idioma(df_comments_us_raw, ["en"])

Total comentarios US (sin filtrar): 1000

--- Filtro de idioma US: solo inglés ---
Distribución de idiomas detectados:
idioma_detectado
en         693
unknown     26
cy          24
so          23
de          19
fr          19
id          19
ro          16
tr          14
af          14
Name: count, dtype: int64

Comentarios antes del filtro: 1000
Comentarios después del filtro (en): 693


In [13]:
# Guardar comentarios filtrados
comments_us = df_comments_us.to_dict(orient="records")

with open("../datos/processed/tiktok/tiktok_lululemon_comentarios_US.json", "w", encoding="utf-8") as f:
    json.dump(comments_us, f, ensure_ascii=False, indent=4)

df_comments_us.to_csv("../datos/processed/tiktok/tiktok_lululemon_comentarios_US.csv", index=False, encoding="utf-8-sig")

print(f"Guardado: {len(df_comments_us)} comentarios en inglés")
print("  ../datos/processed/tiktok/tiktok_lululemon_comentarios_US.csv / .json")

Guardado: 693 comentarios en inglés
  ../datos/processed/tiktok/tiktok_lululemon_comentarios_US.csv / .json


---
## 6. Extracción de comentarios - Europa

Se extraen comentarios de los top vídeos europeos y se filtran por idioma del país:
- GB → inglés
- ES → español
- FR → francés
- DE → alemán

In [14]:
# Mapeo de país de búsqueda → idiomas permitidos en los comentarios
IDIOMAS_POR_PAIS = {
    "GB": ["en"],
    "ES": ["es"],
    "FR": ["fr"],
    "DE": ["de"],
}

# Extraer comentarios por país para poder filtrar por idioma
all_comments_eu = []

for code, nombre in EU_COUNTRIES:
    # Seleccionar top vídeos de este país
    df_pais = df_europe[df_europe["location_search"] == code].copy()
    if len(df_pais) == 0:
        print(f"\n{nombre} ({code}): sin vídeos, saltando.")
        continue
    
    urls_pais = df_pais.head(5)["post_url"].dropna().tolist()  # top 5 por país
    if not urls_pais:
        print(f"\n{nombre} ({code}): sin URLs válidas, saltando.")
        continue
    
    print(f"\n{'='*50}")
    print(f"{nombre} ({code}): extrayendo comentarios de {len(urls_pais)} vídeos...")
    
    run_eu = client.actor("clockworks/tiktok-comments-scraper").call(
        run_input={
            "postURLs": urls_pais,
            "maxComments": MAX_COMMENTS,
            "maxRepliesPerComment": MAX_REPLIES,
        }
    )
    
    items_comments = list(client.dataset(run_eu["defaultDatasetId"]).iterate_items())
    print(f"  Comentarios extraídos (sin filtrar): {len(items_comments)}")
    
    if len(items_comments) == 0:
        continue
    
    df_temp = pd.DataFrame(items_comments)
    
    # Filtrar por idioma del país
    idiomas = IDIOMAS_POR_PAIS.get(code, ["en"])
    print(f"  Filtro de idioma: {idiomas}")
    df_filtrado = filtrar_comentarios_por_idioma(df_temp, idiomas)
    
    # Marcar el país de origen
    df_filtrado["pais_busqueda"] = code
    all_comments_eu.append(df_filtrado)

# Juntar todos los comentarios europeos
if all_comments_eu:
    df_comments_eu = pd.concat(all_comments_eu, ignore_index=True)
else:
    df_comments_eu = pd.DataFrame()

print(f"\n{'='*50}")
print(f"Total comentarios Europa (filtrados por idioma): {len(df_comments_eu)}")
if "pais_busqueda" in df_comments_eu.columns:
    print(df_comments_eu["pais_busqueda"].value_counts())


Reino Unido (GB): extrayendo comentarios de 5 vídeos...


[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> Status: RUNNING, Message: 
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.579Z ACTOR: Pulling container image of build ESKgmPC1krhzsvwG3 from registry.
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.581Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.694Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.696Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.906Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:F66w5uEW7hEySGv0W] -> 2026-03-26T12:45:47.906Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:

  Comentarios extraídos (sin filtrar): 500
  Filtro de idioma: ['en']
Distribución de idiomas detectados:
idioma_detectado
en         316
unknown     29
de          27
ro          13
cy          12
fr          11
so          11
es          11
af           9
ca           8
Name: count, dtype: int64

Comentarios antes del filtro: 500
Comentarios después del filtro (en): 316

España (ES): extrayendo comentarios de 5 vídeos...


[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> Status: RUNNING, Message: 
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.617Z ACTOR: Pulling container image of build ESKgmPC1krhzsvwG3 from registry.
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.619Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.661Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.662Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.845Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:V7OftPTTHAqK1jWjZ] -> 2026-03-26T12:46:47.845Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:

  Comentarios extraídos (sin filtrar): 400
  Filtro de idioma: ['es']
Distribución de idiomas detectados:
idioma_detectado
en         298
unknown     24
ro          20
so           7
tl           7
cy           7
it           5
nl           4
et           4
de           3
Name: count, dtype: int64

Comentarios antes del filtro: 400
Comentarios después del filtro (es): 0

Francia (FR): extrayendo comentarios de 5 vídeos...


[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> Status: RUNNING, Message: 
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.559Z ACTOR: Pulling container image of build ESKgmPC1krhzsvwG3 from registry.
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.562Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.730Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.731Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.903Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:U8qxAzNS9zyCAPLuX] -> 2026-03-26T12:48:09.906Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:

  Comentarios extraídos (sin filtrar): 500
  Filtro de idioma: ['fr']
Distribución de idiomas detectados:
idioma_detectado
en         303
unknown     22
es          17
fr          14
so          14
ro          13
cy          13
de          10
tl          10
no           9
Name: count, dtype: int64

Comentarios antes del filtro: 500
Comentarios después del filtro (fr): 14

Alemania (DE): extrayendo comentarios de 5 vídeos...


[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> Status: RUNNING, Message: Starting the crawler.
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:49.819Z ACTOR: Pulling container image of build ESKgmPC1krhzsvwG3 from registry.
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:49.829Z ACTOR: Creating container.
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:49.864Z ACTOR: Starting container.
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:49.864Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:50.093Z Running on architecture: x86_64
[apify.tiktok-comments-scraper runId:QSEknpISjzmrwayJq] -> 2026-03-26T12:48:50.094Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED     

  Comentarios extraídos (sin filtrar): 500
  Filtro de idioma: ['de']
Distribución de idiomas detectados:
idioma_detectado
en         303
de          33
unknown     22
ro          15
cy          14
es          12
fr          11
so           9
nl           8
fi           7
Name: count, dtype: int64

Comentarios antes del filtro: 500
Comentarios después del filtro (de): 33

Total comentarios Europa (filtrados por idioma): 363
pais_busqueda
GB    316
DE     33
FR     14
Name: count, dtype: int64


In [15]:
# Guardar comentarios Europa filtrados
comments_eu = df_comments_eu.to_dict(orient="records")

with open("../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.json", "w", encoding="utf-8") as f:
    json.dump(comments_eu, f, ensure_ascii=False, indent=4)

df_comments_eu.to_csv("../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.csv", index=False, encoding="utf-8-sig")

print(f"Guardado: {len(df_comments_eu)} comentarios filtrados por idioma")
print("  ../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.csv / .json")

Guardado: 363 comentarios filtrados por idioma
  ../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.csv / .json


---
## 7. Resumen final

In [16]:
print("="*60)
print("RESUMEN DE EXTRACCIÓN")
print("="*60)
print(f"\nVídeos encontrados:")
print(f"  US:     {len(df_us)} vídeos")
print(f"  Europa: {len(df_europe)} vídeos")
print(f"\nComentarios extraídos (filtrados por idioma):")
print(f"  US (inglés):  {len(df_comments_us)} comentarios")
print(f"  Europa:       {len(df_comments_eu)} comentarios")
if "pais_busqueda" in df_comments_eu.columns:
    for code, nombre in EU_COUNTRIES:
        n = len(df_comments_eu[df_comments_eu["pais_busqueda"] == code])
        idioma = IDIOMAS_POR_PAIS.get(code, ["?"])[0]
        print(f"    {nombre} ({idioma}): {n}")
print(f"\nArchivos generados:")
print(f"  ../datos/processed/tiktok/tiktok_lululemon_videos_US.csv")
print(f"  ../datos/processed/tiktok/tiktok_lululemon_videos_EUROPE.csv")
print(f"  ../datos/processed/tiktok/tiktok_lululemon_comentarios_US.csv / .json")
print(f"  ../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.csv / .json")

RESUMEN DE EXTRACCIÓN

Vídeos encontrados:
  US:     200 vídeos
  Europa: 200 vídeos

Comentarios extraídos (filtrados por idioma):
  US (inglés):  693 comentarios
  Europa:       363 comentarios
    Reino Unido (en): 316
    España (es): 0
    Francia (fr): 14
    Alemania (de): 33

Archivos generados:
  ../datos/processed/tiktok/tiktok_lululemon_videos_US.csv
  ../datos/processed/tiktok/tiktok_lululemon_videos_EUROPE.csv
  ../datos/processed/tiktok/tiktok_lululemon_comentarios_US.csv / .json
  ../datos/processed/tiktok/tiktok_lululemon_comentarios_EUROPE.csv / .json
